# BP3 Gate 1 — Business Understanding & Policy
**Customer360 Navigator Enterprise Suite — Complaint Escalation / Intervention Prediction**

## Why this notebook exists, and what it honestly scopes
Master Execution Plan Section 5.1 / Section 7 define BP3 as: *"select only a defensible
observed/proxy outcome (e.g. company response = 'closed with monetary relief' or a disputed flag);
run leakage checks before any model sees it; report ROC-AUC, PR-AUC, recall and calibration, never a
bare accuracy figure on an imbalanced target."* This is BP3's very first notebook (Gate 0 -> Gate 1,
not a retroactive gap the way BP1's Gate 1 was) — it grounds that definition against the real,
already-profiled CFPB data (`docs/data_dictionary/RAW_DATA_MANIFEST.md`, `DATA_PROFILE_REPORT.md`,
and BP2's own Gate 1/2/3 real-run findings) before any Gate 3 model-benchmark work is built.

**One of the Master Plan's two named example targets is not supportable by the real data in scope,
and this notebook says so explicitly rather than fabricating a substitute silently:**
- **"A disputed flag"** — NOT buildable. The real CFPB extract's 15-column schema
  (`RAW_DATA_MANIFEST.md` Section 2, re-verified live below) has no `Consumer disputed?` column or
  equivalent. CFPB's public dataset dropped that field in 2017; this project's extract post-dates
  that change. No disputed-flag proxy is invented here.
- **"Company response = closed with monetary relief"** — buildable, and used below. The real CFPB
  extract carries `Company response to consumer` as a structured outcome field with 6 real distinct
  values (already live-enumerated once, for BP2's own Gate 1, and re-verified live again in this
  notebook rather than assumed carried over) — one of which is CFPB's own literal
  `"Closed with monetary relief"` label.

## A real, disclosed overlap with BP2 — not coincidental
BP2 (Customer Friction Classification) and BP3 (this notebook) both derive their target from the
same real `Company response to consumer` field, because it is the only structured outcome field this
extract has. They are kept genuinely distinct, not two names for one model:
- BP2 asks an **ordinal severity** question across all 6 real response values (including
  `"Untimely response"`, which BP2's precedence rule promotes to its own top HIGH_FRICTION class).
- BP3 asks a **binary intervention** question — did the company have to give monetary relief — and
  **excludes** `"Untimely response"` rows entirely (see the target definition below) precisely
  because that CFPB label is already BP2's own primary friction signal; folding it into BP3's target
  too would define two BPs' targets from the same rows on the same field, blurring their scopes
  instead of keeping them separable for BP7's later decision-engine inputs.

This is an explicit, reviewable design choice (see ASSUMPTIONs in the written policy), not a proven-
necessary one — open to revision at Gate 3 if the real class balance below makes it untenable.

## Purpose
Produces BP3's Gate 1 output exactly as Section 8 (6-Gate Governance SOP) defines it: a policy
artifact recording the target definition, leakage rules, and ASSUMPTIONs — verified against the
real, profiled data, not invented. Gate 1's own exit criterion ("No target leakage possible by
construction") is verified live below.

## Standing rules this notebook follows
- **Execution boundary** (Section 12.2): Claude wrote this notebook; it does not run it. You run it
  on your own machine, and the real, live-checked results below become this project's Gate 1 policy
  record for BP3.
- **Zero-fabrication** (Section 12.1): every check below runs against the real file in `data/raw/`.
  No category label, count, or distinct-value string in this notebook is asserted from memory — all
  are read live from the real CFPB CSV, including the `Tags` re-check described next.
- **ECOA / Regulation B (Master Plan Section 9)** — BP3 is one of the four BPs this section maps to
  ECOA/Reg B. BP1/BP2's own Gate 1 notebooks stated "No Protected-Class Field in Scope," true of the
  15-column schema's *named* fields — but BP2's later Gate 3 real run found that `Tags` (one of those
  15 columns) actually contains real demographic-adjacent values (`servicemember`, `older american`).
  This notebook does not repeat the now-superseded blanket claim: it live-re-enumerates `Tags` itself
  (Section 5 below) and, given a demographic-adjacent field is confirmed present, excludes `Tags`
  from BP3's candidate feature set outright at Gate 1 — a conservative choice made now specifically
  so BP3 never needs the disparate-impact-style testing Section 9 would otherwise require before
  Gate 6, rather than deferring that compliance burden.
- **WARP**: `configure_performance()` first. This is the same real 1,048,575-row CFPB file BP2 reads
  in full at Gate 1 — loaded via `pl.scan_csv` (lazy) exactly as BP2's Gate 1 does, not
  `pandas.read_csv`.
- **HYPER**: reuses `src/taxonomy/taxonomy_mapper.CFPB_DTYPES` (real column dtypes, no
  re-derivation), `src/utils/bp1_config_sync.py` (generic marker-based config read/write, already
  reused unmodified by BP2), and BP2 Gate 1's own ID-like-keyword check for "no persistent
  customer/consumer identifier."
- **No BANKING77 integration**: the Master Plan's BP table states BP3 does **not** integrate
  BANKING77 (`Integrates BANKING77? = NO`) — BANKING77's schema is `text,category` only regardless
  (verified live by BP1/BP2's own Gate 1 notebooks) and carries no complaint-outcome field BP3 could
  use even if it wanted to. Not loaded in this notebook at all.
- **Idempotent**: re-running this notebook overwrites `configs/bp3_complaint_escalation_prediction.yaml`
  (front matter only) and this notebook's own `policy.json` artifact in place.
- **PROJECT_STRUCTURE_LOCKED.md rule #3**: same resolver as every other notebook in this project.

## Target definition (Business Understanding, Master Plan Section 5.1/7, BP3)
`intervention_required` — binary:
- **1 (INTERVENTION_REQUIRED)**: `Company response to consumer == "Closed with monetary relief"` —
  CFPB's own explicit, observed label for a complaint the company had to give monetary relief for.
  This is the Master Plan's own named example proxy outcome, used as-is.
- **0 (NO_INTERVENTION_REQUIRED)**: `Company response to consumer` in
  `{"Closed with explanation", "Closed with non-monetary relief"}` — resolved without the company
  being compelled to provide monetary relief.
- **Excluded from the trainable set** (no defensible label, or a deliberate scope boundary — real
  row counts for each reason are live-computed in Section 6, not estimated):
  - `"In progress"` — unresolved; there is no real observed outcome to label yet (same reasoning as
    BP2's own `EXCLUDED_PENDING` class).
  - `"Untimely response"` — CFPB's timeliness-failure label; excluded here because it is already
    BP2's own primary friction signal on this same field (see "A real, disclosed overlap with BP2"
    above), not because it lacks a defensible outcome.
  - Null `Company response to consumer` (2 rows in the whole real extract, per
    `DATA_PROFILE_REPORT.md`) — unknown outcome (same reasoning as BP2's own `EXCLUDED_UNKNOWN`
    class).

## Outputs (both written, idempotent overwrite-in-place)
- `configs/bp3_complaint_escalation_prediction.yaml` — `target_definition`, `leakage_rules`,
  `assumptions`, `status` written to the front-matter section (existing gate blocks, if any,
  preserved verbatim)
- `notebooks/bp3_complaint_escalation_prediction/artifacts/policy.json` — the Section 8 Gate 1
  output artifact, with the live category-enumeration and class-balance results embedded

## Prerequisites
`01_data_acquisition_profiling.ipynb` should have been real-run at least once (Sprint 1), though this
notebook re-verifies everything it needs independently rather than trusting that artifact blindly.

## If a structural check below fails
It raises `AssertionError` with the failing check named. A failing leakage check in particular must
never be worked around — if a barred field's real correlation with the target turns out to make it
look like a shortcut into the label itself, that is a real modeling risk (inflated evaluation
metrics) and must be fixed in the feature set, not in this check.

In [ ]:
# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
import os
import sys
from pathlib import Path


def _find_project_root(marker_filename: str = "PROJECT_STRUCTURE_LOCKED.md") -> Path:
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        candidate = Path(env_override)
        if (candidate / marker_filename).exists():
            return candidate
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {candidate} but {marker_filename} was not found there. "
            "Fix the environment variable rather than removing this check."
        )

    start = Path.cwd()
    current = start
    for _ in range(8):
        if (current / marker_filename).exists():
            return current
        if current.parent == current:
            break
        current = current.parent

    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker_filename in filenames:
            return Path(depth_root)

    raise RuntimeError(
        "Could not resolve PROJECT_ROOT. Set the C360_PROJECT_ROOT environment variable to the "
        "Customer360_Navigator_Enterprise_Suite folder, or run this notebook from inside the project tree "
        "(expected at notebooks/bp3_complaint_escalation_prediction/)."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
CONFIGS_DIR = PROJECT_ROOT / "configs"
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp3_complaint_escalation_prediction" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP - configure_performance() FIRST, before any heavy/BLAS-backed import
# ============================================================
from utils.performance_setup import configure_performance  # noqa: E402

WARP_SUMMARY = configure_performance(project_root=PROJECT_ROOT, verbose=True)

# ============================================================
# SECTION 3: Heavy imports + flush-forcing print override (LESSONS_LEARNED_APPLIED.md #12)
# ============================================================
import builtins  # noqa: E402
import functools  # noqa: E402
import json  # noqa: E402
import warnings  # noqa: E402
from datetime import datetime, timezone  # noqa: E402

import polars as pl  # noqa: E402

from taxonomy.taxonomy_mapper import CFPB_DTYPES  # noqa: E402

warnings.filterwarnings("ignore")
print = functools.partial(builtins.print, flush=True)

CFPB_PATH = DATA_RAW_DIR / "cfpb_complaints.csv"

# ============================================================
# SECTION 4: Structural checks - real CFPB schema re-verified live (not asserted from
# RAW_DATA_MANIFEST.md's prior documentation alone), plus the same "no persistent
# customer/consumer identifier" check BP2 Gate 1 already established (HYPER, reused verbatim).
# ============================================================
cfpb_columns = list(CFPB_DTYPES.keys())
EXPECTED_CFPB_COLUMNS = [
    "Date received",
    "Product",
    "Sub-product",
    "Issue",
    "Sub-issue",
    "Company public response",
    "Company",
    "State",
    "ZIP code",
    "Tags",
    "Submitted via",
    "Date sent to company",
    "Company response to consumer",
    "Timely response?",
    "Complaint ID",
]
ID_LIKE_KEYWORDS = ("customer", "consumer id", "person", "account number", "ssn", "email", "phone")
suspected_customer_id_columns = [c for c in cfpb_columns if any(kw in c.lower() for kw in ID_LIKE_KEYWORDS)]
print(f"[OK] Real CFPB columns ({len(cfpb_columns)}): {cfpb_columns}")
print(
    f"[OK] Columns matching customer/consumer-identifier keywords: {suspected_customer_id_columns or 'NONE'}"
)

# ============================================================
# SECTION 5: Live enumeration of the real candidate target/compliance fields - the actual label
# strings (not just distinct counts) are fetched live here, never asserted from memory or copied
# from BP2's own already-documented values.
# ============================================================
cfpb_lazy = pl.scan_csv(CFPB_PATH, schema_overrides=CFPB_DTYPES)

company_response_counts = (
    cfpb_lazy.group_by("Company response to consumer")
    .agg(pl.len().alias("n"))
    .sort("n", descending=True)
    .collect()
)
timely_response_counts = (
    cfpb_lazy.group_by("Timely response?").agg(pl.len().alias("n")).sort("n", descending=True).collect()
)
tags_counts = cfpb_lazy.group_by("Tags").agg(pl.len().alias("n")).sort("n", descending=True).collect()

total_rows = cfpb_lazy.select(pl.len()).collect().item()

print(f"[OK] Real CFPB row count (live): {total_rows:,}")
print("[OK] Live 'Company response to consumer' distinct values + real counts:")
print(company_response_counts)
print("[OK] Live 'Timely response?' distinct values + real counts:")
print(timely_response_counts)
print("[OK] Live 'Tags' distinct values + real counts (ECOA/Reg B re-check, Master Plan Section 9):")
print(tags_counts)

DEMOGRAPHIC_ADJACENT_KEYWORDS = ("servicemember", "older american", "veteran")
tags_values_lower = [str(v).lower() for v in tags_counts["Tags"].to_list() if v is not None]
demographic_adjacent_tags_found = [
    v
    for v in tags_counts["Tags"].to_list()
    if v is not None and any(kw in str(v).lower() for kw in DEMOGRAPHIC_ADJACENT_KEYWORDS)
]
print(f"[OK] Demographic-adjacent 'Tags' values found live: {demographic_adjacent_tags_found or 'NONE'}")

# ============================================================
# SECTION 6: Live class balance for BP3's binary target definition (see the notebook's markdown
# cell for the full target definition and the reasoning behind each exclusion category).
# ============================================================
response_col = "Company response to consumer"
n_intervention_required = (
    cfpb_lazy.filter(pl.col(response_col) == "Closed with monetary relief").select(pl.len()).collect().item()
)
n_no_intervention_required = (
    cfpb_lazy.filter(
        pl.col(response_col).is_in(["Closed with explanation", "Closed with non-monetary relief"])
    )
    .select(pl.len())
    .collect()
    .item()
)
n_excluded_in_progress = (
    cfpb_lazy.filter(pl.col(response_col) == "In progress").select(pl.len()).collect().item()
)
n_excluded_untimely_response = (
    cfpb_lazy.filter(pl.col(response_col) == "Untimely response").select(pl.len()).collect().item()
)
n_excluded_null_response = cfpb_lazy.filter(pl.col(response_col).is_null()).select(pl.len()).collect().item()
n_trainable = n_intervention_required + n_no_intervention_required
n_excluded_total = n_excluded_in_progress + n_excluded_untimely_response + n_excluded_null_response

target_class_balance = {
    "n_intervention_required": int(n_intervention_required),
    "n_no_intervention_required": int(n_no_intervention_required),
    "n_trainable_total": int(n_trainable),
    "n_excluded_in_progress": int(n_excluded_in_progress),
    "n_excluded_untimely_response": int(n_excluded_untimely_response),
    "n_excluded_null_response": int(n_excluded_null_response),
    "n_excluded_total": int(n_excluded_total),
    "positive_class_ratio_of_trainable": (
        round(n_intervention_required / n_trainable, 4) if n_trainable else None
    ),
}
print(f"[OK] Live BP3 target class balance: {target_class_balance}")
print(
    f"[OK] Row accounting check: trainable ({n_trainable:,}) + excluded ({n_excluded_total:,}) "
    f"= {n_trainable + n_excluded_total:,} vs real total ({total_rows:,})"
)

# ============================================================
# SECTION 7: Assemble the Gate 1 policy (target definition, leakage rules, assumptions)
# ============================================================
policy = {
    "bp_id": "bp3",
    "bp_name": "bp3_complaint_escalation_prediction",
    "gate": 1,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "target_definition": {
        "primary_target": "intervention_required",
        "primary_target_description": "Binary: 1 if the real `Company response to consumer` value "
        "is 'Closed with monetary relief' (CFPB's own explicit "
        "observed intervention outcome, the Master Plan's own named "
        "example proxy); 0 if 'Closed with explanation' or 'Closed "
        "with non-monetary relief'. 'In progress', 'Untimely "
        "response', and null-response rows are excluded from the "
        "trainable set - see exclusion_reasons below and this "
        "notebook's markdown cell for the full reasoning.",
        "exclusion_reasons": {
            "In progress": "Unresolved - no real observed outcome to label yet (mirrors BP2's own "
            "EXCLUDED_PENDING class).",
            "Untimely response": "Excluded deliberately, not for lack of a defensible outcome: this "
            "CFPB label is already BP2's own primary friction-severity signal "
            "on this same field. Including it in BP3's target too would define "
            "two BPs' targets from the same rows on the same field. Open "
            "design choice, flagged for Gate 3 review, not proven necessary.",
            "null Company response to consumer": "Unknown outcome, 2 rows in the whole real extract "
            "per DATA_PROFILE_REPORT.md (mirrors BP2's own "
            "EXCLUDED_UNKNOWN class).",
        },
        "disputed_flag_not_used": "The Master Plan's other named example target ('a disputed flag') "
        "is not buildable: the real 15-column CFPB schema (re-verified "
        "live in Section 4) has no `Consumer disputed?` column or "
        "equivalent - CFPB dropped that field from its public dataset in "
        "2017 and this extract post-dates that change. Not substituted "
        "with an invented proxy.",
        "feature_variable_candidates": "Structured CFPB fields only: Product, Sub-product, Issue, "
        "Sub-issue, State, Submitted via, Company (frequency-"
        "encoded, HYPER-reusing BP2's own pattern in "
        "src/features/bp2_friction_features.py). Final feature set "
        "is a Gate 2/3 decision. No BANKING77 field (BP3 does not "
        "integrate BANKING77 per the Master Plan's BP table).",
        "train_test_split_source": "CFPB ships no pre-defined train/test split - BP3 will use a "
        "fresh random split of the real, trainable-only CFPB rows, "
        "stratified by intervention_required, random_state=42.",
    },
    "leakage_rules": [
        "'Company response to consumer' defines the target and must NEVER also be used as an input "
        "feature - it is not traditional train/test leakage, it is the label itself.",
        "'Timely response?' is NOT used to define this target (only 'Company response to consumer' "
        "is), but is barred from the feature set anyway as a conservative default - it is "
        "conceptually close to 'escalation' and BP2 Gate 2 already found real rows where it "
        "disagrees with 'Company response to consumer' (1,963 rows), so its true leakage risk for "
        "THIS target is unverified. Deferred to Gate 3 for an explicit test before any relaxation, "
        "the same pattern BP2 Gate 3 used for 'Company public response.'",
        "'Date received' and 'Date sent to company' are barred from the feature set - a computed "
        "response-time duration was found to leak the timeliness-derived half of BP2's own target "
        "on this identical data (BP2 Gate 3 finding); the same risk is assumed to apply here until "
        "Gate 3 tests it directly, not assumed safe by default.",
        "'Tags' is barred from the feature set - live-re-checked in Section 5 rather than trusting "
        "BP1/BP2 Gate 1's now-superseded 'no protected-class field' claim, and found (or "
        "re-confirmed found, matching BP2 Gate 3's real finding) to contain demographic-adjacent "
        "values. BP3 is ECOA/Reg B-mapped (Master Plan Section 9); excluding 'Tags' outright avoids "
        "needing disparate-impact-style testing before Gate 6.",
        "'Complaint ID' and 'ZIP code' are barred as a pure identifier and a fine-grained "
        "quasi-identifier respectively - never used as model features, matching BP2's own "
        "BARRED_COLUMNS precedent (HYPER-consistent).",
        "No BANKING77 data is used for BP3 at all (Master Plan BP table: Integrates BANKING77? = "
        "NO) - not loaded in this notebook.",
        "No CFPB row may appear in both the train and test split of BP3's fresh random split - "
        "enforced structurally at Gate 3 (single stratified split call, not a manual/ad hoc merge).",
    ],
    "assumptions": [
        "No 'disputed' flag exists in this real CFPB extract (re-verified live in Section 4); BP3's "
        "target therefore uses the Master Plan's other named example, 'company response = closed "
        "with monetary relief', not a disputed-flag proxy.",
        "'Untimely response' rows are excluded from BP3's binary target as an explicit, disclosed "
        "design choice (not a data limitation) to keep BP2's and BP3's targets separable on the "
        "same source field - open to Gate 3 review if this makes the trainable class balance "
        "unworkable.",
        "'In progress' and null-response rows are excluded as having no observed/resolved outcome "
        "yet, mirroring BP2's own EXCLUDED_PENDING / EXCLUDED_UNKNOWN precedent on the same field.",
        "'Timely response?' and the two Date fields are barred from BP3's feature set at Gate 1 as "
        "a conservative default, not yet proven necessary - deferred to Gate 3 for an explicit "
        "leakage/ablation test before any relaxation, the same practice BP2 Gate 3 used for "
        "'Company public response.'",
        "src/utils/bp1_config_sync.py is reused unmodified for BP3's own config file - already "
        "fully generic, parameterized by config_path, no BP1-specific logic (already reused "
        "unmodified by BP2 too).",
    ],
    "compliance_touchpoint": {
        "requirement": "ECOA / Regulation B - Fair Lending & Adverse Action (Master Plan Section 9; "
        "BP3 is one of the four BPs this section explicitly maps to ECOA/Reg B)",
        "statement": "The real CFPB extract's 15 named columns carry no demographic or "
        "protected-class field by name. However, 'Tags' (one of those 15 columns) was "
        "found - live, in Section 5 above, re-confirming BP2 Gate 3's own real finding "
        "on the same data - to contain demographic-adjacent values. Per Master Plan "
        "Section 9's own rule ('if any demographic-adjacent field is present, "
        "disparate-impact-style testing runs before Gate 6'), BP3 excludes 'Tags' from "
        "its feature set outright at this Gate rather than deferring that compliance "
        "burden. With 'Tags' excluded, no remaining candidate feature is demographic-"
        "adjacent, so disparate-impact-style testing is not required for the feature "
        "set BP3 actually uses - stated as a scoping decision, not a claim that no "
        "protected-class-adjacent data exists anywhere in the source file.",
    },
    "live_checks": {
        "cfpb_row_count": total_rows,
        "cfpb_columns": cfpb_columns,
        "cfpb_columns_match_manifest": cfpb_columns == EXPECTED_CFPB_COLUMNS,
        "suspected_customer_id_columns": suspected_customer_id_columns,
        "company_response_to_consumer_distribution": company_response_counts.to_dicts(),
        "timely_response_distribution": timely_response_counts.to_dicts(),
        "tags_distribution": tags_counts.to_dicts(),
        "demographic_adjacent_tags_found": demographic_adjacent_tags_found,
        "target_class_balance": target_class_balance,
    },
}

# ============================================================
# SECTION 8: Write outputs (idempotent overwrite-in-place)
# ============================================================
policy_json_path = ARTIFACTS_DIR / "policy.json"
with open(policy_json_path, "w", encoding="utf-8") as f:
    json.dump(policy, f, indent=2, default=str)
print(f"\n[SAVED] {policy_json_path.relative_to(PROJECT_ROOT)}")

bp3_config_path = CONFIGS_DIR / "bp3_complaint_escalation_prediction.yaml"

# BP3 reuses BP1's marker-based config-sync helpers as-is (src/utils/bp1_config_sync.py) - already
# generic, already reused unmodified by BP2. Gate 1 here owns only the front-matter section below;
# every later gate's block (once written) is preserved verbatim regardless of position or order.
# See LESSONS_LEARNED_APPLIED.md #20 for the real incident this pattern was built to prevent.
from utils.bp1_config_sync import read_existing_gate_block_markers, write_front_matter  # noqa: E402

_existing_gate_markers = read_existing_gate_block_markers(bp3_config_path)
_status_suffix = ""
for _gate_num, _gate_label in ((2, "Gate 2"), (3, "Gate 3"), (4, "Gate 4"), (5, "Gate 5")):
    if any(_gate_label in _m for _m in _existing_gate_markers):
        _status_suffix += f"_gate{_gate_num}_confirmed"

bp3_config_text = f"""# Per-BP config - filled in at Gate 1 (Business Understanding & Policy)
# Gate 1 owns bp_id through random_state below via write_front_matter() (src/utils/bp1_config_sync.py,
# reused as-is from BP1/BP2 - fully generic, parameterized by config_path); Gates 2-5 each own exactly
# one marker-delimited block appended after it via write_gate_block() - do not hand-edit either
# section, re-run the owning notebook instead.
bp_id: "bp3"
bp_name: "bp3_complaint_escalation_prediction"
status: "gate1_confirmed{_status_suffix}"   # not_started|gate1|gate2|gate3|gate4|gate5|gate6_complete
target_definition:
  primary_target: "intervention_required"
  primary_target_description: "Binary: 1 if real 'Company response to consumer' == 'Closed with
    monetary relief' (CFPB's own explicit observed outcome, Master Plan's named example proxy);
    0 if 'Closed with explanation' or 'Closed with non-monetary relief'. 'In progress', 'Untimely
    response', and null-response rows excluded from the trainable set - see policy.json
    exclusion_reasons for the full live-computed row accounting."
  disputed_flag_not_used: "No 'Consumer disputed?' column exists in this real CFPB extract
    (CFPB dropped it from the public dataset in 2017) - re-verified live, not assumed."
  feature_variable_candidates: "Structured CFPB fields only: Product, Sub-product, Issue,
    Sub-issue, State, Submitted via, Company (frequency-encoded). No BANKING77 field (BP3 does
    not integrate BANKING77 per the Master Plan's BP table)."
  train_test_split_source: "Fresh random split of the real, trainable-only CFPB rows, stratified
    by intervention_required, random_state=42."
leakage_rules:
  - "'Company response to consumer' defines the target and must never also be used as an input
     feature."
  - "'Timely response?' barred from the feature set as a conservative default - real disagreement
     with 'Company response to consumer' found on this data by BP2 Gate 2 (1,963 rows); deferred
     to Gate 3 for an explicit leakage test."
  - "'Date received' and 'Date sent to company' barred - a computed response-time duration leaked
     BP2's own target on this identical data (BP2 Gate 3 finding); assumed to apply here until
     Gate 3 tests it directly."
  - "'Tags' barred - live-re-checked (not assumed from BP1/BP2 Gate 1's now-superseded claim) and
     found to contain demographic-adjacent values, matching BP2 Gate 3's real finding. BP3 is
     ECOA/Reg B-mapped (Master Plan Section 9)."
  - "'Complaint ID' and 'ZIP code' barred as identifier / quasi-identifier, matching BP2's
     BARRED_COLUMNS precedent."
  - "No BANKING77 data used at all for BP3 (Master Plan BP table: Integrates BANKING77? = NO)."
  - "No CFPB row may appear in both train and test splits - enforced structurally at Gate 3."
assumptions:
  - "No 'disputed' flag exists in this real CFPB extract (re-verified live); the Master Plan's
     other named example target ('closed with monetary relief') is used instead."
  - "'Untimely response' rows are excluded from BP3's binary target as an explicit, disclosed
     design choice to keep BP2's and BP3's targets separable on the same source field - open to
     Gate 3 review, not proven necessary."
  - "'In progress' and null-response rows excluded as having no observed/resolved outcome yet,
     mirroring BP2's own EXCLUDED_PENDING / EXCLUDED_UNKNOWN precedent."
  - "'Timely response?' and the two Date fields barred from the feature set at Gate 1 as a
     conservative default, deferred to Gate 3 for an explicit leakage/ablation test."
  - "src/utils/bp1_config_sync.py reused unmodified for BP3's own config file - already fully
     generic, no BP1-specific logic."
random_state: 42
"""
write_front_matter(bp3_config_path, bp3_config_text)
print(f"[SAVED] {bp3_config_path.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 9: Structural integrity checks - raise AssertionError, never silently pass
# ============================================================
checks = {
    "cfpb_schema_matches_manifest": cfpb_columns == EXPECTED_CFPB_COLUMNS,
    "no_customer_identifier_column_in_cfpb_schema": len(suspected_customer_id_columns) == 0,
    "company_response_to_consumer_enumerated": company_response_counts.height > 0,
    "timely_response_enumerated": timely_response_counts.height > 0,
    "tags_enumerated": tags_counts.height > 0,
    "target_row_accounting_matches_total": (n_trainable + n_excluded_total) == total_rows,
    "at_least_one_row_per_target_class": n_intervention_required > 0 and n_no_intervention_required > 0,
    "policy_json_written": policy_json_path.exists(),
    "bp3_config_yaml_written": bp3_config_path.exists(),
}

print("\n=== INTEGRITY CHECKS ===")
for name, passed in checks.items():
    status = "[PASS]" if passed else "[FAIL]"
    print(f"{status} {name}")
    assert passed, f"[CHECK FAILED] {name}"

print(
    "\n[ALL CHECKS PASSED] BP3 Gate 1 complete - target defined (intervention_required, binary), "
    "'disputed flag' honestly ruled out, 'Untimely response' overlap with BP2 disclosed and "
    "excluded by design, 'Tags' re-checked live and barred as demographic-adjacent, real class "
    "balance live-computed. Proceed to BP3 Gate 2 (Data Verification & Feature/Taxonomy "
    "Engineering) next."
)